# Datamine REGMOW Process Example

This notebook demonstrates how to configure and run the **`regmow`** process wrapper in `dmstudio`.

## Process Description

## REGMOW

Create a regular cell model for pit optimization.

* **Note** (This can be performed on any type of Datamine block model.):

### Input Files:

* **in1** (Block Model Prototype):
  Input model prototype file, defining the new model origin, number of cells and cell
  sizes. This is typically set up by process **[PROTOM](<protom.md>)**. The fields
  required are **XMORIG, YMORIG, ZMORIG, NX, NY, NZ, XINC, YINC, ZINC** (all implicit) and
  **IJK, XC, YC, and ZC** (all explicit).
  Required=Yes

* **in2** (Block Model):
  Input model file for conversion. This must have the fields **XMORIG, YMORIG, ZMORIG,
  NX, NY, NZ** (implicit) and **IJK, XC, YC and ZC** (explicit). **XINC, YINC and ZINC**
  must exist as either explicit (sub-cells permitted) or implicit (no sub-cells). There
  must also be at least one explicit numeric data field, to be specified as **F1**. The
  records may be in any order, but speed is greatly increased if they are in **IJK**
  order.
  Required=Yes

### Output Files:

* **out** (Block Model File):
  Output model file. This will have the model parameters of the input prototype file on
  IN1 , and may contain up to five averaged fields ( **F1** -**F5**). It will also contain
  the fields **OREVOL, WASVOL** and **AIRVOL**.
  Required=Yes

### Fields:

* **bltype** (Any : IN2):
  Field in **IN2** which contains a specific value **AIRVAL** when the sub-cell is air.
  Default=Undefined
  Required=No

* **f1_f5** (Numeric : IN2):
  Explicit numeric fields to be averaged. F1 is mandatory.
  Default=Undefined
  Required=No

### Parameters:

* **airval**:
  The value of the **BLTYPE** field that will be used to recognize air blocks and
  sub-blocks in the input model.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **cutoff1**:
  If the value in field **F1** is below this, the sub-cell is treated as waste.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **restrict**:
  Set to 1 if only the blocks on the input model are to be reported in the regularized
  model. Default is 0.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  >=2 display for each input cell or sub-cell that intersects with an output model cell;
  **IJK1, IJK, NUMMET, XC, YC, ZC, VOLP, VOLT, F1** [IJK of input and output cell,
  sub-cell no., input cell centre, volume intersected, total volume to date in output
  cell, **F1** value] and for each output cell, **IJK, WASVOL, OREVOL** , the **F1** name
  and the **F1** value. (0).
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('regmow')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute regmow
print("Running regmow...")
dm_cmd.regmow(
    inmods_i=['optional'],  # required
    out_o='t_regmow_out',  # required
    # bltype_f='optional',  # optional
    # f1_f5_f='optional',  # optional
    # airval_p='optional',  # optional
    # cutoff1_p='optional',  # optional
    # restrict_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("regmow execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_regmow_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")